🟦 INTRODUCTION

# Traffic Data Cleaning

This notebook focuses on cleaning and preparing the raw traffic dataset. It involves removing unnecessary columns, extracting useful time-based features, encoding categorical variables, and preparing the data for machine learning.

## Import Libraries

This section imports the pandas library used for data manipulation and analysis.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/traffic_data.csv")

### Preview Dataset

This displays the first few rows of the dataset to understand its structure and contents.

In [3]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


### Drop Unnecessary Columns

This removes columns that are not needed for the analysis or model training.

In [4]:
df = df.drop(columns=['holiday', 'weather_description'])

In [5]:
df.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918


### Feature Engineering (Date & Time)

This converts the date column to datetime format and extracts useful features such as hour, day of week, month, and year.

In [6]:
df['date_time'] = pd.to_datetime(df['date_time'])

# Extract time-based features
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month'] = df['date_time'].dt.month
df['year'] = df['date_time'].dt.year

### Create Additional Features

This creates new features to indicate weekends and rush hour periods, which are important for traffic prediction.

In [7]:
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

df['is_rush_hour'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 9 or 16 <= x <= 18) else 0
)

In [8]:
df.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,day_of_week,month,year,is_weekend,is_rush_hour
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2012,0,1
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2012,0,0
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2012,0,0
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2012,0,0
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2012,0,0


### Drop Original Date Column

This removes the original date column since relevant features have already been extracted.

In [9]:
df = df.drop(columns=['date_time'])

### Clean Weather Categories

This groups less common weather conditions into a single category for simplicity.

In [10]:
df['weather_main'] = df['weather_main'].replace({
    'Squall': 'Other',
    'Smoke': 'Other'
})

### Encode Categorical Variables

#### Encode Weather Data

This converts categorical weather data into numerical format using one-hot encoding.

In [11]:
df = pd.get_dummies(df, columns=['weather_main'], drop_first=True)

In [12]:
df.head()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume,hour,day_of_week,month,year,is_weekend,is_rush_hour,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Other,weather_main_Rain,weather_main_Snow,weather_main_Thunderstorm
0,288.28,0.0,0.0,40,5545,9,1,10,2012,0,1,True,False,False,False,False,False,False,False,False
1,289.36,0.0,0.0,75,4516,10,1,10,2012,0,0,True,False,False,False,False,False,False,False,False
2,289.58,0.0,0.0,90,4767,11,1,10,2012,0,0,True,False,False,False,False,False,False,False,False
3,290.13,0.0,0.0,90,5026,12,1,10,2012,0,0,True,False,False,False,False,False,False,False,False
4,291.14,0.0,0.0,75,4918,13,1,10,2012,0,0,True,False,False,False,False,False,False,False,False


### Create Target Variable

This creates a binary target variable (traffic_level) based on the median traffic volume.

In [13]:
threshold = df['traffic_volume'].median()

df['traffic_level'] = df['traffic_volume'].apply(
    lambda x: 1 if x > threshold else 0
)

In [14]:
df.head()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume,hour,day_of_week,month,year,is_weekend,...,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Other,weather_main_Rain,weather_main_Snow,weather_main_Thunderstorm,traffic_level
0,288.28,0.0,0.0,40,5545,9,1,10,2012,0,...,True,False,False,False,False,False,False,False,False,1
1,289.36,0.0,0.0,75,4516,10,1,10,2012,0,...,True,False,False,False,False,False,False,False,False,1
2,289.58,0.0,0.0,90,4767,11,1,10,2012,0,...,True,False,False,False,False,False,False,False,False,1
3,290.13,0.0,0.0,90,5026,12,1,10,2012,0,...,True,False,False,False,False,False,False,False,False,1
4,291.14,0.0,0.0,75,4918,13,1,10,2012,0,...,True,False,False,False,False,False,False,False,False,1


### Drop Original Traffic Volume

This removes the original traffic volume column after creating the target variable.

In [15]:
df = df.drop(columns=['traffic_volume'])

In [16]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   temp                       48204 non-null  float64
 1   rain_1h                    48204 non-null  float64
 2   snow_1h                    48204 non-null  float64
 3   clouds_all                 48204 non-null  int64  
 4   hour                       48204 non-null  int32  
 5   day_of_week                48204 non-null  int32  
 6   month                      48204 non-null  int32  
 7   year                       48204 non-null  int32  
 8   is_weekend                 48204 non-null  int64  
 9   is_rush_hour               48204 non-null  int64  
 10  weather_main_Clouds        48204 non-null  bool   
 11  weather_main_Drizzle       48204 non-null  bool   
 12  weather_main_Fog           48204 non-null  bool   
 13  weather_main_Haze          48204 non-null  boo

### Convert Boolean Columns

This converts boolean values into integers for compatibility with machine learning models.

In [17]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

In [18]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   temp                       48204 non-null  float64
 1   rain_1h                    48204 non-null  float64
 2   snow_1h                    48204 non-null  float64
 3   clouds_all                 48204 non-null  int64  
 4   hour                       48204 non-null  int32  
 5   day_of_week                48204 non-null  int32  
 6   month                      48204 non-null  int32  
 7   year                       48204 non-null  int32  
 8   is_weekend                 48204 non-null  int64  
 9   is_rush_hour               48204 non-null  int64  
 10  weather_main_Clouds        48204 non-null  int64  
 11  weather_main_Drizzle       48204 non-null  int64  
 12  weather_main_Fog           48204 non-null  int64  
 13  weather_main_Haze          48204 non-null  int

### Save Cleaned Dataset

This saves the cleaned dataset for use in model training.

In [19]:
df.to_csv("../data/processed/cleaned_traffic_data.csv", index=False)

### Summary of Data Cleaning Process

In this notebook, several preprocessing steps were applied to transform the raw traffic dataset into a structured format suitable for machine learning.

Unnecessary columns such as `holiday` and `weather_description` were removed because they either contained redundant information or were not useful for prediction. The `date_time` column was converted into multiple meaningful features (hour, day of week, month, and year) since machine learning models cannot directly interpret raw datetime values.

Additional features like `is_weekend` and `is_rush_hour` were created to capture real-world traffic patterns, as traffic behavior is often influenced by time and day. After extracting these features, the original `date_time` column was dropped to avoid redundancy.

Weather categories were simplified by grouping less frequent values, and then encoded into numerical format using one-hot encoding, making them suitable for the model.

A target variable (`traffic_level`) was created by converting continuous traffic volume into a binary classification (High or Low traffic), which aligns with the goal of the prediction system. The original `traffic_volume` column was then removed since it is no longer needed.

Finally, boolean values were converted into integers and the cleaned dataset was saved for use in model training.

Overall, these steps ensure that the dataset is clean, consistent, and properly structured for building an effective machine learning model.